# MC inclusive-jet JEC check

Compare the selected inclusive-jet eta distributions before and after jet energy corrections (JEC) in several reconstructed-jet pT intervals. The raw and corrected inputs are `hRecoInclusiveJetRawPtEtaLabUnflipped` and `hRecoInclusiveJetPtEtaLabUnflipped`. For each pT interval, the notebook makes three figures: raw/corrected yield overlays, corrected/raw ratios, and corrected/raw overlays with the Pb-going-to-p-going ratio.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository or a subdirectory.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetOptStat(0)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.python.histogram_io import load_histogram, resolve_direction_file
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import set_1d_style, save_canvas


In [ ]:
GENERATOR = 'embedding'              # 'embedding' or 'pythia'
FILE_STEM = 'jetId'                  # production selection stem
PT_RANGES = ((30., 50.), (50., 80.), (80., 120.), (120., 180.), (180., 300.), (300., 500.))
ETA_RANGE = None                     # None means the full eta axis
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'mc_jet_JEC_check'
SAVE_PNG = False
Y_RANGE = None                       # e.g. (1e-2, 1e7), or None for automatic
CORRECTED_RAW_RANGE = (0.5, 1.5)
DIRECTION_RATIO_RANGE = (0.8, 1.2)

hist_names = {
    'raw': 'hRecoInclusiveJetRawPtEtaLabUnflipped',
    'corrected': 'hRecoInclusiveJetPtEtaLabUnflipped',
}
files = {direction: resolve_direction_file(BASE_DIR, GENERATOR, direction, FILE_STEM)
         for direction in ('Pbgoing', 'pgoing')}
missing = [path for path in files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing MC input file(s):\n' + '\n'.join(map(str, missing)))
histograms = {direction: {kind: load_histogram(path, name) for kind, name in hist_names.items()}
             for direction, path in files.items()}
print('Inputs:')
for direction, path in files.items():
    print(f'  {direction}: {path}')


In [ ]:
_objects = []

def project(direction, kind, pt_range):
    return project_semantic_th2(
        histograms[direction][kind], 'eta', pt_range,
        name=f'{direction}_{kind}_eta_{pt_range[0]:g}_{pt_range[1]:g}',
    )

def draw_overlay(canvas, hists, title, y_title, *, logy=False, y_range=None):
    canvas.cd()
    canvas.SetLogy(logy)
    first = True
    for index, (label, hist) in enumerate(hists.items()):
        set_1d_style(hist, index)
        hist.SetTitle(f'{title};#eta_{{lab}};{y_title}')
        hist.GetXaxis().SetRangeUser(-5., 5.)
        if y_range is not None:
            hist.SetMinimum(y_range[0]); hist.SetMaximum(y_range[1])
        hist.Draw('E' if first else 'E SAME')
        first = False
    legend = ROOT.TLegend(0.58, 0.70, 0.88, 0.88)
    legend.SetBorderSize(0); legend.SetFillStyle(0)
    for label, hist in hists.items():
        legend.AddEntry(hist, label, 'lep')
    legend.Draw()
    canvas.Update()
    return legend

def draw_three_panel(pt_range):
    tag = f'pt_{pt_range[0]:g}_{pt_range[1]:g}'
    raw = {d: project(d, 'raw', pt_range) for d in files}
    corrected = {d: project(d, 'corrected', pt_range) for d in files}
    corrected_raw = {d: ratio_to_nominal(corrected[d], raw[d], name=f'{d}_corrected_over_raw_{tag}') for d in files}
    direction_ratio = ratio_to_nominal(corrected_raw['Pbgoing'], corrected_raw['pgoing'], name=f'Pbgoing_over_pgoing_{tag}')
    _objects.extend((*raw.values(), *corrected.values(), *corrected_raw.values(), direction_ratio))

    canvas1 = ROOT.TCanvas(f'c_jec_yields_{tag}', '', 800, 700)
    draw_overlay(canvas1, {'Pb-going raw': raw['Pbgoing'], 'Pb-going corrected': corrected['Pbgoing'],
                           'p-going raw': raw['pgoing'], 'p-going corrected': corrected['pgoing']},
                 f'MC inclusive jets, {pt_range[0]:g} < p_{{T}} < {pt_range[1]:g} GeV', 'Yield', logy=True, y_range=Y_RANGE)
    save_canvas(canvas1, OUTPUT_DIR / f'{tag}_raw_corrected.pdf', save_png=SAVE_PNG)

    canvas2 = ROOT.TCanvas(f'c_jec_corrected_raw_{tag}', '', 800, 700)
    draw_overlay(canvas2, {'Pb-going corrected/raw': corrected_raw['Pbgoing'], 'p-going corrected/raw': corrected_raw['pgoing']},
                 f'JEC response, {pt_range[0]:g} < p_{{T}} < {pt_range[1]:g} GeV', 'Corrected / Raw', y_range=CORRECTED_RAW_RANGE)
    canvas2.SetGridy(True); save_canvas(canvas2, OUTPUT_DIR / f'{tag}_corrected_over_raw.pdf', save_png=SAVE_PNG)

    canvas3 = ROOT.TCanvas(f'c_jec_direction_{tag}', '', 800, 700)
    draw_overlay(canvas3, {'Pb-going corrected/raw': corrected_raw['Pbgoing'], 'p-going corrected/raw': corrected_raw['pgoing'],
                           'Pb-going / p-going': direction_ratio},
                 f'JEC response and beam direction, {pt_range[0]:g} < p_{{T}} < {pt_range[1]:g} GeV', 'Ratio', y_range=DIRECTION_RATIO_RANGE)
    canvas3.SetGridy(True); save_canvas(canvas3, OUTPUT_DIR / f'{tag}_direction_comparison.pdf', save_png=SAVE_PNG)
    return canvas1, canvas2, canvas3

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
figures = [draw_three_panel(pt_range) for pt_range in PT_RANGES]
print(f'Wrote {3 * len(figures)} figures to {OUTPUT_DIR}')
